# ch09 小模型训练与规模判定（练习）

> 把 `____` 换成正确的代码。每道题下方的 `assert` 会立刻告诉你对不对。

**本章目标**：学会在动手训练之前，先把「参数量 → 内存 → 耗时」三张账算清楚。

## 三个核心公式

| # | 公式 | 验证标准 |
|---|---|---|
| ① | 参数量恒等式（embeddings + 每层 Encoder + pooler） | 与实测对比，**相对误差 0.0000%** |
| ② | 训练内存 `= 16 B/参数 + 激活` | 逐张量精确统计 |
| ③ | 激活 `∝ batch（线性）× g(seq)（超线性）` | 6 组 (batch, seq) 实测 |

## 量内存的正确姿势

| 手法 | 精度 | 用途 |
|---|---|---|
| `tensor_bytes()` | **精确** | 参数 / 梯度 / 优化器状态 |
| `activation_bytes()` | **精确** | 激活 |
| `RSS` | 噪声极大 | **本章一次都不用**（内存池复用会让它偏几十倍） |

In [ ]:
import gc
import time
from copy import deepcopy

import pandas as pd
import torch
from transformers import BertConfig, BertModel

torch.set_num_threads(1)          # 耗时标定要稳定 —— 多线程会让数字剧烈波动
SEED = 42
MB = 1024 ** 2
GB = 1024 ** 3

## 脚手架（已给好，直接调用）

In [ ]:
# ---- 五档规模：从"玩具"一路到 BERT-large，全部用 Config 随机初始化 ----
SCALES = {
    "tiny":  dict(vocab_size=1000,  hidden_size=64,   num_hidden_layers=2,
                  num_attention_heads=2,  intermediate_size=256,  max_position_embeddings=64),
    "small": dict(vocab_size=4000,  hidden_size=128,  num_hidden_layers=4,
                  num_attention_heads=4,  intermediate_size=512,  max_position_embeddings=128),
    "mid":   dict(vocab_size=8000,  hidden_size=256,  num_hidden_layers=6,
                  num_attention_heads=8,  intermediate_size=1024, max_position_embeddings=256),
    "base":  dict(vocab_size=21128, hidden_size=768,  num_hidden_layers=12,
                  num_attention_heads=12, intermediate_size=3072, max_position_embeddings=512),
    "large": dict(vocab_size=21128, hidden_size=1024, num_hidden_layers=24,
                  num_attention_heads=16, intermediate_size=4096, max_position_embeddings=512),
}

SCALE_BATCHES = [(1, 32), (2, 32), (4, 32), (2, 64), (2, 128), (8, 32)]

FREEZE_PLAN = [("完全不冻结", False, 0), ("只冻结 emb", True, 0),
               ("emb + 前 2 层", True, 2), ("emb + 前 4 层", True, 4)]


def bert_params(V, d, L, f, max_pos, type_vocab=2):
    """BERT 参数量恒等式（fp32，含全部 bias 与 LayerNorm）。

    拆成三块：
      embeddings: V*d + max_pos*d + type_vocab*d + 2*d   （三个 Embedding + 一层 LayerNorm）
      每层 Encoder: 4d^2 + 4d   （q/k/v/attn-out 四个 d*d 矩阵，各带 d 个 bias）
                    + 2d        （attention 后的 LayerNorm）
                    + 2*f*d + f （FFN 第一层，升维）
                    + f*d + d   （FFN 第二层，降维）
                    + 2d        （FFN 后的 LayerNorm）
                    => 4d^2 + 2*f*d + 9*d + f
      pooler: d*d + d
    """
    emb = d * (V + max_pos + type_vocab) + 2 * d
    layer = 4 * d * d + 2 * f * d + 9 * d + f
    pooler = d * d + d
    return {"emb": emb, "layer": layer, "pooler": pooler,
            "total": emb + L * layer + pooler}


def params_of(name):
    """按档位名算参数量 —— SCALES 的 key 与恒等式形参名不同，这里显式挑出来。"""
    kw = SCALES[name]
    return bert_params(kw["vocab_size"], kw["hidden_size"], kw["num_hidden_layers"],
                       kw["intermediate_size"], kw["max_position_embeddings"])


def make_model(name, dropout=0.1):
    cfg = BertConfig(type_vocab_size=2, hidden_dropout_prob=dropout,
                     attention_probs_dropout_prob=dropout, **SCALES[name])
    return cfg, BertModel(cfg)


def make_batch(name, B, S, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = torch.randint(0, SCALES[name]["vocab_size"], (B, S), generator=g)
    return ids, torch.ones_like(ids)


def tensor_bytes(iterable):
    """按实际 dtype 的 element_size 累加字节数 —— 这是唯一精确的口径。

    （不要用 RSS 去量小模型：Python 内存池复用会让它偏差几十倍。）
    """
    return sum(t.numel() * t.element_size() for t in iterable if torch.is_tensor(t))


def opt_state_bytes(opt):
    """AdamW 的 state 是 {step, exp_avg, exp_avg_sq}，逐条数出来。"""
    return sum(t.numel() * t.element_size()
               for st in opt.state.values() for t in st.values() if torch.is_tensor(t))


def activation_bytes(model, ids, am):
    """激活内存 = 反向传播所需的全部「保存张量」。

    用 saved_tensors_hooks 拦截 forward 期间被保存的每一个张量，
    按 data_ptr 去重后累加 —— 这比估公式精确，也比量 RSS 干净。
    """
    saved = []

    def pack(t):
        saved.append(t)
        return t

    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        model(ids, attention_mask=am).last_hidden_state.pow(2).mean().backward()

    seen, tot = set(), 0
    for t in saved:
        if t.data_ptr() in seen:
            continue
        seen.add(t.data_ptr())
        tot += t.numel() * t.element_size()
    return tot


def micro_batches(ids_all, am_all, B_micro):
    """把整批切成若干微批 —— 梯度累积要按同样的口径切。"""
    n = ids_all.shape[0] // B_micro
    return [(ids_all[i * B_micro:(i + 1) * B_micro],
             am_all[i * B_micro:(i + 1) * B_micro]) for i in range(n)]


def freeze_params(model, freeze_emb, n_layer):
    """冻结 embeddings（可选）与前 n_layer 层，返回冻结的张量个数。"""
    to_freeze = list(model.embeddings.parameters()) if freeze_emb else []
    for layer in model.encoder.layer[:n_layer]:
        to_freeze.extend(layer.parameters())
    for p in to_freeze:
        p.requires_grad = False
    return len(to_freeze)


def fwd_fn(model, ids, am):
    return model(ids, attention_mask=am).last_hidden_state


def step_fn(model, opt, ids, am):
    fwd_fn(model, ids, am).pow(2).mean().backward()
    opt.step()
    opt.zero_grad()


def timed(fn, n=3):
    fn()                                    # warmup，避免把首次编译/分配算进去
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n


def timed_fwd_ms(model, ids, am, n=3):
    """纯推理（no_grad）下的前向耗时，毫秒。"""
    with torch.no_grad():
        model(ids, attention_mask=am)
        t0 = time.perf_counter()
        for _ in range(n):
            model(ids, attention_mask=am)
        return (time.perf_counter() - t0) / n * 1000

## 任务 1：参数量恒等式

用恒等式算出 tiny 档参数量。

In [ ]:
p_tiny = params_of("tiny")["total"]
assert p_tiny == 172480, p_tiny
print(f"tiny 档参数量 {p_tiny:,}")

## 任务 2：五档规模与组件占比

算全部五档，并拆出 embeddings / Encoder / pooler 的占比。

In [ ]:
rows = []
for name, kw in SCALES.items():
    p = params_of(name)
    rows.append({"档位": name,
                 "参数量": p["total"],
                 "Emb%": round(p["emb"] / p["total"] * 100, 1),
                 "Layer%": round(p["layer"] * kw["num_hidden_layers"] / p["total"] * 100, 1),
                 "Pooler%": round(p["pooler"] / p["total"] * 100, 1)})
param_table = pd.DataFrame(rows).set_index("档位")
print(param_table.to_string())

## 任务 3：恒等式 vs 实测

实例化 tiny / small / mid，数一遍 `sum(p.numel())` 对账。

In [ ]:
check = {}
for name in ("tiny", "small", "mid"):
    _, model = make_model(name)
    actual = sum(p.numel() for p in model.parameters())
    expect = params_of(name)["total"]
    check[name] = {"恒等式": f"{expect:,}", "实测": f"{actual:,}",
                   "相对误差%": round((actual - expect) / actual * 100, 4)}
    del model
    gc.collect()
assert all(v["相对误差%"] == 0.0 for v in check.values()), check
print(pd.DataFrame(check).T.to_string())

## 任务 4：内存四件套

激活 / 参数 / 梯度 / 优化器状态，四项分开量。注意**激活必须最先量**（它内部要做一次 backward）。

In [ ]:
mem_rows = []
for name in ("tiny", "small", "mid"):
    _, model = make_model(name)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    ids, am = make_batch(name, 2, 32)
    n_param = sum(p.numel() for p in model.parameters())
    act_b = activation_bytes(model, ids, am) / MB
    opt.step()                       # step 一次，AdamW 才会真正建出 exp_avg / exp_avg_sq
    param_b = tensor_bytes(model.parameters()) / MB
    grad_b = tensor_bytes(p.grad for p in model.parameters() if p.grad is not None) / MB
    opt_b = opt_state_bytes(opt) / MB
    total_b = param_b + grad_b + opt_b + act_b
    mem_rows.append({"档位": name, "参数MB": round(param_b, 2), "梯度MB": round(grad_b, 2),
                     "优化器MB": round(opt_b, 2), "激活MB": round(act_b, 2),
                     "合计MB": round(total_b, 2),
                     "B/参数": round(total_b * MB / n_param, 1)})
    del model, opt
    gc.collect()
mem_table = pd.DataFrame(mem_rows).set_index("档位")
print(mem_table.to_string())

## 任务 5：优化器状态精确账

验证它是否恰好等于 `2P × 4` 字节。

In [ ]:
P_mid = params_of("mid")["total"]
D_MID = SCALES["mid"]["hidden_size"]
_, mid_model = make_model("mid")
mid_model.train()
mid_opt = torch.optim.AdamW(mid_model.parameters(), lr=1e-4)
ids5, am5 = make_batch("mid", 2, 32)
mid_model(ids5, attention_mask=am5).last_hidden_state.pow(2).mean().backward()
mid_opt.step()
n_grad = sum(p.numel() for p in mid_model.parameters() if p.grad is not None)
n_no_grad = sum(p.numel() for p in mid_model.parameters() if p.grad is None)
measured = opt_state_bytes(mid_opt)
assert n_grad + n_no_grad == P_mid
assert n_no_grad == D_MID * D_MID + D_MID, n_no_grad
assert measured == 2 * n_grad * 4 + len(mid_opt.state) * 4, (measured, n_grad, len(mid_opt.state))
print(f"总参数 {P_mid:,} | 有梯度 {n_grad:,} | 无梯度 {n_no_grad:,}"
      f"（= pooler 的 {D_MID}² + {D_MID}）")
print(f"优化器状态 {measured:,} B = 2 × {n_grad:,} × 4 B + {len(mid_opt.state)} × 4 B（每条的 step）")

## 任务 6：激活的缩放律

扫 6 组 `(batch, seq)`，看激活怎么随二者变化。

In [ ]:
scale_rows = []
_, m6 = make_model("mid")
m6.train()
for B, S in SCALE_BATCHES:
    m6.zero_grad()
    ids6, am6 = make_batch("mid", B, S)
    ab6 = activation_bytes(m6, ids6, am6)
    scale_rows.append({"batch": B, "seq": S, "B×S": B * S,
                       "激活MB": round(ab6 / MB, 2),
                       "每token B": round(ab6 / (B * S))})
act_table = pd.DataFrame(scale_rows)
print(act_table.to_string(index=False))

## 任务 7：单步耗时标定

分别测纯前向与整步，算倍率。

In [ ]:
time_rows = []
for name in ("tiny", "small", "mid"):
    _, model = make_model(name)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    ids7, am7 = make_batch(name, 2, 32)
    n_param = sum(p.numel() for p in model.parameters())
    fwd_ms = timed(lambda: fwd_fn(model, ids7, am7)) * 1000
    step_ms = timed(lambda: step_fn(model, opt, ids7, am7)) * 1000
    time_rows.append({"档位": name, "参数量": n_param, "前向ms": round(fwd_ms, 2),
                      "整步ms": round(step_ms, 2), "倍率": round(step_ms / fwd_ms, 2)})
    del model, opt
    gc.collect()
print(pd.DataFrame(time_rows).to_string(index=False))

## 任务 8：base 档 —— 算得出来，跑得动吗

先算训练态内存，再测前向耗时。

In [ ]:
base_train_gb = params_of("base")["total"] * 16 / GB
assert round(base_train_gb, 2) == 1.52, base_train_gb
print(f"base 档参数量 {params_of('base')['total']:,}"
      f"（{params_of('base')['total'] / 1e6:.1f} M）| 训练态理论内存 {base_train_gb:.2f} GB")

base_cfg, base_model = make_model("base", dropout=0.0)
base_model.eval()
for B, S in [(1, 32), (8, 128)]:
    ms = timed_fwd_ms(base_model, *make_batch("base", B, S))
    print(f"  前向 batch={B} seq={S}：{ms:.1f} ms")

## 任务 9：规模判定 —— 8 GB 能训多大

给定预算反查可行档位。

In [ ]:
budget_rows = []
BUDGET_GB = 8.0
for name in SCALES:
    need_gb = params_of(name)["total"] * 16 / GB
    budget_rows.append({"档位": name,
                        "参数量M": round(params_of(name)["total"] / 1e6, 1),
                        "训练态GB": round(need_gb, 2),
                        "装得下": "是" if need_gb < BUDGET_GB else "否"})
print(pd.DataFrame(budget_rows).to_string(index=False))

## 任务 10：梯度累积 —— 梯度等价吗

把 `B=8` 切成 4 份 `B=2` 累加，验证梯度是否与整批一致。

> ⚠️ **关键**：每份的 loss 要除以累积步数，否则梯度会差 4 倍。

In [ ]:
torch.manual_seed(SEED)
_, model_a = make_model("mid", dropout=0.0)
model_b = deepcopy(model_a)
ids_all, am_all = make_batch("mid", 8, 32, seed=7)
model_a.train()
model_b.train()
# ---- A：整批 B=8 一次算完 ----
fwd_fn(model_a, ids_all, am_all).pow(2).mean().backward()
grad_full = [p.grad.clone() for p in model_a.parameters() if p.grad is not None]
for i in range(4):
    ids_i, am_i = ids_all[i * 2:(i + 1) * 2], am_all[i * 2:(i + 1) * 2]
    (fwd_fn(model_b, ids_i, am_i).pow(2).mean() / 4).backward()
grad_acc = [p.grad.clone() for p in model_b.parameters() if p.grad is not None]

max_abs = max((a - b).abs().max().item() for a, b in zip(grad_full, grad_acc))
scale = max(g.abs().max().item() for g in grad_full)
rel = max_abs / scale
assert rel < 1e-5, rel
print(f"张量组数 {len(grad_full)} | 最大绝对差 {max_abs:.3e} | 梯度量级 {scale:.4f} "
      f"| 相对差 {rel:.3e}")

## 任务 11：梯度累积 —— 省多少激活

量出累积时的激活峰值，与整批对比。

In [ ]:
torch.manual_seed(SEED)
_, m_full = make_model("mid", dropout=0.0)
m_full.train()
act_full = activation_bytes(m_full, ids_all, am_all) / MB

torch.manual_seed(SEED)
_, m_acc = make_model("mid", dropout=0.0)
m_acc.train()
acts = [activation_bytes(m_acc, bi, ba) / MB for bi, ba in micro_batches(ids_all, am_all, 2)]
peak = max(acts)
print(f"整批 B=8 激活 {act_full:.2f} MB | 累积 B=2×4 峰值 {peak:.2f} MB "
      f"→ 降到 {peak / act_full * 100:.1f}%")

## 任务 12：冻结策略的收益

对比「冻结 emb」与「冻结层」分别省了哪一份内存。

In [ ]:
freeze_rows = []
for tag, freeze_emb, n_layer in FREEZE_PLAN:
    _, fm = make_model("mid")
    freeze_params(fm, freeze_emb, n_layer)
    fm.train()
    ids12, am12 = make_batch("mid", 2, 32)
    act_f = activation_bytes(fm, ids12, am12) / MB
    grad_f = tensor_bytes(p.grad for p in fm.parameters() if p.grad is not None) / MB
    freeze_rows.append({"冻结策略": tag,
                        "可训练": sum(p.numel() for p in fm.parameters() if p.requires_grad),
                        "梯度MB": round(grad_f, 2), "激活MB": round(act_f, 2)})
    del fm
    gc.collect()
print(pd.DataFrame(freeze_rows).to_string(index=False))

## 任务 13：精度换内存

把模型转成 bf16 / fp16，看参数内存变化。

In [ ]:
dtype_rows = []
for dtype, tag in [(torch.float32, "fp32"), (torch.bfloat16, "bf16"), (torch.float16, "fp16")]:
    m13 = deepcopy(m_full).to(dtype)
    b13 = tensor_bytes(m13.parameters()) / MB
    dtype_rows.append({"精度": tag, "参数MB": round(b13, 2)})
    del m13
gc.collect()
print(pd.DataFrame(dtype_rows).to_string(index=False))

## 自查清单

- [ ] 参数量恒等式能默写（embeddings / 每层 Encoder / pooler 三块）
- [ ] 知道 16 B/参数 = 权重 4 + 梯度 4 + Adam 状态 8
- [ ] 知道 `AdamW` 的 state 惰性创建，不 `step()` 量不到
- [ ] 能解释激活在 batch 方向线性、seq 方向超线性
- [ ] 知道梯度累积要 `loss / accumulation_steps`
- [ ] 能分辨「冻结 emb」和「冻结层」省的**不是同一份**内存
- [ ] 背下经验式：**参数量（亿）× 1.6 ≈ 全参训练显存（GB）**